# Práctica Final — Análisis de Sentimientos con LLMs
## Modelo: `microsoft/Phi-3-mini-4k-instruct`

---

**Asignatura:** Búsqueda y Análisis de Información  
**Autores:** Juan Battaglio Quintana, Juan Guasp Timoner, Alejandro Parra Purroy  
**Fecha:** Mayo 2026

---

### Descripción del modelo

**Phi-3-mini-4k-instruct** es un modelo de lenguaje de Microsoft lanzado en **abril de 2024**, con **3,8 mil millones de parámetros** y una ventana de contexto de **4096 tokens**. Pertenece a la familia Phi-3, diseñada para maximizar el rendimiento en tareas de razonamiento con un tamaño reducido. La variante `instruct` ha sido ajustada para seguir instrucciones en formato conversacional.

| Característica | Valor |
|---|---|
| Nombre | `microsoft/Phi-3-mini-4k-instruct` |
| Empresa | Microsoft |
| Parámetros | 3,8 B |
| Fecha de publicación | Abril 2024 |
| Ventana de contexto | 4096 tokens |
| Arquitectura | Decoder-only Transformer (causal LM) con grouped-query attention |
| Tipo de acceso | Público (no requiere licencia ni token) |
| Licencia | MIT |
| Tensor type recomendado | float16 |

**Motivo de elección:** Phi-3-mini ofrece un equilibrio interesante entre tamaño (3,8 B) y calidad de razonamiento, con acceso completamente público. Funciona en GPU T4 sin necesidad de cuantización. Sirve como referente intermedio entre Gemma 2 (2,6 B) y Mistral 7B (7,2 B) del grupo.

---

### Tarea elegida por el grupo

**Clasificación de sentimiento de reseñas de producto** en tres clases: **positiva**, **negativa**, **neutra**.

Reseña de prueba (texto fijo común a los tres prompts y a los tres modelos del grupo):

> *"El producto llegó antes de lo esperado y el embalaje estaba perfecto. Sin embargo, después de usarlo tres días, dejó de funcionar correctamente. El servicio de atención al cliente no respondió a mis mensajes. Muy decepcionante."*

**Etiqueta esperada (gold):** `negativa`. La reseña reconoce dos aspectos positivos puntuales (rapidez de envío y embalaje) pero el sentimiento dominante es claramente negativo: fallo del producto, ausencia de soporte y cierre con "muy decepcionante".

---

### Diseño experimental

- **3 prompts** (base, plantilla clara, razonamiento y contraste).
- **5 repeticiones** por prompt → 15 ejecuciones totales.
- **Temperatura > 0** (`temperature=0.7`, `top_p=0.9`) para forzar variabilidad.
- **Métricas objetivas y subjetivas** definidas más abajo.

### Índice

0. Instalación de dependencias  
1. Configuración del entorno  
2. Carga del modelo  
3. Definición del problema y prompts  
4. Función de inferencia  
5. Ejecución del experimento  
6. Métricas objetivas  
7. Métricas subjetivas  
8. Tabla comparativa final  
9. Análisis de variabilidad  
10. Exportar resultados  
11. Conclusiones


---
## 0. Instalación de dependencias

Ejecuta esta celda si alguna de las librerías necesarias no está instalada en tu entorno. En **Google Colab** se recomienda ejecutarla siempre. En **entorno local** solo es necesaria la primera vez.

> **Nota:** Después de la instalación puede ser necesario **reiniciar el kernel** antes de continuar.


In [ ]:
# Instalar versiones específicas compatibles entre sí para Phi-3-mini
# transformers 4.44.x + huggingface_hub < 0.30 evitan el StrictDataclassDefinitionError

!pip install \
    "transformers==4.44.2" \
    "huggingface_hub==0.24.6" \
    "accelerate>=0.26.0" \
    sentencepiece protobuf -q

# Verificar versiones instaladas
import importlib
for pkg in ['torch', 'transformers', 'huggingface_hub', 'accelerate', 'sentencepiece']:
    try:
        mod = importlib.import_module(pkg)
        print(f'  {pkg}: {getattr(mod, "__version__", "instalado")}')
    except ImportError:
        print(f'  {pkg}: NO instalado')

print("\nREINICIA el runtime de Colab antes de continuar (Entorno de ejecución → Reiniciar sesión)")


---
## 1. Configuración del entorno

Importamos las librerías necesarias y detectamos el hardware disponible.


In [ ]:
import torch
import time
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Detectar dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_libre = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM total: {vram_total:.1f} GB")
    print(f"VRAM libre: {vram_libre:.1f} GB")
else:
    print("ADVERTENCIA: No se detectó GPU. El modelo tardará considerablemente más en CPU.")

print(f"\nPyTorch version: {torch.__version__}")


---
## 2. Carga del modelo

Cargamos `microsoft/Phi-3-mini-4k-instruct` desde Hugging Face. El modelo ocupa aproximadamente **7,6 GB en float16** en VRAM, lo que cabe holgadamente en una T4 (15 GB).

A diferencia de Gemma 2 y Mistral 7B (que son *gated* y requieren token de Hugging Face), Phi-3 es de **acceso público**: no hace falta autenticarse.

> **Nota:** La primera ejecución descargará los pesos del modelo (~7 GB). Las ejecuciones posteriores los cargarán desde caché local.


In [ ]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

print(f"Cargando tokenizador de '{MODEL_ID}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Cargando modelo (float16, device_map=auto)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Creando pipeline de generación de texto...")
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("\n✅ Pipeline listo.")
print(f"   Arquitectura : {pipe.model.config.model_type}")
print(f"   Device       : {pipe.model.device}")
print(f"   Dtype        : {pipe.model.dtype}")
print(f"   Parámetros   : {sum(p.numel() for p in pipe.model.parameters()) / 1e6:.1f} M")


### 2.1 Test de conexión

Verificamos que el modelo responde correctamente con una pregunta neutral antes de lanzar el experimento.


In [ ]:
test_messages = [{"role": "user", "content": "¿Cuál es la capital de Francia? Responde en una frase."}]

test_output = pipe(test_messages, max_new_tokens=50, do_sample=False)
test_respuesta = test_output[0]["generated_text"][-1]["content"]

print("=== TEST DE CONEXIÓN ===")
print(f"Pregunta : ¿Cuál es la capital de Francia?")
print(f"Respuesta: {test_respuesta}")
print("========================")


---
## 3. Definición del problema y prompts

La reseña es la misma para los tres prompts y para los tres modelos del grupo. Lo único que cambia es la **estrategia de prompting**.

### Prompt 1 — Base
Instrucción directa y mínima, sin estructura formal ni ejemplos. Sirve como línea base para la comparación.

### Prompt 2 — Plantilla estructurada
Sigue el esquema enseñado en clase: **Tarea + Contexto + Restricciones + Formato de salida + Criterio de calidad**. Proporciona información explícita sobre cómo responder.

### Prompt 3 — Razonamiento paso a paso
Solicita al modelo que analice el texto en etapas, contraste alternativas y justifique su conclusión. Favorece la transparencia del proceso de decisión.


In [ ]:
RESENA = (
    "El producto llegó antes de lo esperado y el embalaje estaba perfecto. "
    "Sin embargo, después de usarlo tres días, dejó de funcionar correctamente. "
    "El servicio de atención al cliente no respondió a mis mensajes. "
    "Muy decepcionante."
)

# Etiqueta de oro definida por el grupo
ETIQUETA_GOLD = "negativa"

# ---------- Prompt 1: BASE ----------
PROMPT_BASE = f'''Clasifica la siguiente reseña como positiva, negativa o neutra:
"{RESENA}"'''

# ---------- Prompt 2: PLANTILLA CLARA ----------
PROMPT_PLANTILLA = f'''Tarea: Clasifica la reseña de un producto como positiva, negativa o neutra.
Contexto: Eres un sistema de análisis de opiniones para una tienda online.
La clasificación se usará para mejorar el servicio al cliente.
Restricciones:
- Responde únicamente con una de estas etiquetas: positiva, negativa, neutra.
- Añade una justificación de máximo 2 frases.
- No uses información externa a la reseña.
Formato de salida:
Clasificación: [etiqueta]
Justificación: [máximo 2 frases]
Criterio de calidad: La clasificación debe reflejar el sentimiento predominante
del texto, no solo aspectos aislados.
Reseña: "{RESENA}"'''

# ---------- Prompt 3: RAZONAMIENTO Y CONTRASTE ----------
PROMPT_RAZONAMIENTO = f'''Analiza la siguiente reseña de producto paso a paso y determina su clasificación.
Reseña: "{RESENA}"
Sigue estos pasos:
1. Identifica los aspectos positivos mencionados en la reseña.
2. Identifica los aspectos negativos mencionados en la reseña.
3. Considera las tres posibles clasificaciones: positiva, negativa y neutra.
4. Compara el peso de los argumentos a favor de cada una.
5. Escribe una conclusión final justificada con la clasificación elegida.
Formato de respuesta:
Aspectos positivos: ...
Aspectos negativos: ...
Análisis de alternativas: ...
Conclusión y clasificación final: ...'''

PROMPTS = {
    "1_base": PROMPT_BASE,
    "2_plantilla": PROMPT_PLANTILLA,
    "3_razonamiento": PROMPT_RAZONAMIENTO,
}

print(f"Reseña       : {RESENA}")
print(f"Etiqueta gold: {ETIQUETA_GOLD}")
print(f"Prompts      : {list(PROMPTS.keys())}")


### 3.1 Vista previa de los tres prompts


In [ ]:
for nombre, prompt_text in PROMPTS.items():
    print(f"\n{'='*60}")
    print(f"PROMPT: {nombre.upper()}")
    print('='*60)
    print(prompt_text)


---
## 4. Parámetros y función de inferencia

Configuramos los hiperparámetros de generación. La temperatura > 0 es un **requisito obligatorio** de la práctica para poder estudiar la variabilidad entre ejecuciones.

> **Nota sobre coordinación con el resto del grupo:** los tres notebooks (Gemma 2, Mistral 7B y Phi-3) usan idénticos parámetros de generación: `temperature=0.7`, `top_p=0.9`, `do_sample=True`. Así, las diferencias observadas se atribuyen al modelo y al prompt, no a la configuración de muestreo.


In [ ]:
# Verificar que el modelo está en GPU antes de lanzar el experimento
device_modelo = next(model.parameters()).device
print(f"Modelo en: {device_modelo}")
if str(device_modelo) == 'cpu':
    print("⚠ ADVERTENCIA: el modelo está en CPU — activa T4 GPU en Colab")
else:
    vram_usada = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM usada por el modelo: {vram_usada:.1f} GB")

GEN_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
    "max_new_tokens": 200,
}

# Para el prompt de razonamiento permitimos respuestas más largas
GEN_PARAMS_LARGO = {**GEN_PARAMS, "max_new_tokens": 400}

N_REPETICIONES = 5
BATCH_SIZE = 4  # número de prompts procesados en paralelo en GPU

print(f"\nTotal llamadas: {len(PROMPTS) * N_REPETICIONES}")
print(f"Batch size    : {BATCH_SIZE}")


---
## 5. Ejecución del experimento

Ejecutamos todas las combinaciones: **3 prompts × 5 repeticiones = 15 llamadas**.

Los resultados se guardan en un DataFrame y se exportan a CSV al finalizar.

> Tiempo estimado en T4: ~1-2 minutos.


In [ ]:
# Pre-generar todos los inputs como lista plana para procesarlos en batch
all_inputs = []
all_metadata = []

for prompt_nombre, prompt_text in PROMPTS.items():
    for rep in range(1, N_REPETICIONES + 1):
        all_inputs.append([{"role": "user", "content": prompt_text}])
        all_metadata.append({
            "prompt": prompt_nombre,
            "repeticion": rep,
            "temperature": GEN_PARAMS["temperature"],
            "top_p": GEN_PARAMS["top_p"],
            "gold": ETIQUETA_GOLD,
        })

print(f"Inputs preparados: {len(all_inputs)} | Batch size: {BATCH_SIZE}")
print("Ejecutando experimento en GPU...")

# Procesamos los inputs en dos tandas según max_new_tokens (corto vs largo)
# para no malgastar tokens en los prompts cortos
inputs_cortos  = [inp for inp, m in zip(all_inputs, all_metadata) if m["prompt"] != "3_razonamiento"]
meta_cortos    = [m for m in all_metadata if m["prompt"] != "3_razonamiento"]
inputs_largos  = [inp for inp, m in zip(all_inputs, all_metadata) if m["prompt"] == "3_razonamiento"]
meta_largos    = [m for m in all_metadata if m["prompt"] == "3_razonamiento"]

t_inicio_total = time.time()

outputs_cortos = pipe(inputs_cortos, batch_size=BATCH_SIZE, **GEN_PARAMS) if inputs_cortos else []
outputs_largos = pipe(inputs_largos, batch_size=BATCH_SIZE, **GEN_PARAMS_LARGO) if inputs_largos else []

t_total = time.time() - t_inicio_total
n_total = len(inputs_cortos) + len(inputs_largos)
t_medio = t_total / n_total if n_total else 0

# Reconstruir resultados emparejando outputs con metadata
resultados = []
for output, meta in zip(outputs_cortos, meta_cortos):
    respuesta = output[0]["generated_text"][-1]["content"].strip()
    resultados.append({**meta, "respuesta": respuesta, "tiempo_s": round(t_medio, 2)})
for output, meta in zip(outputs_largos, meta_largos):
    respuesta = output[0]["generated_text"][-1]["content"].strip()
    resultados.append({**meta, "respuesta": respuesta, "tiempo_s": round(t_medio, 2)})

df_resultados = pd.DataFrame(resultados).sort_values(["prompt", "repeticion"]).reset_index(drop=True)

print(f"\n✅ Experimento completado en {t_total:.1f} s ({t_total/60:.1f} min)")
print(f"   Tiempo medio por llamada: {t_medio:.1f} s")
df_resultados[["prompt", "repeticion", "tiempo_s"]]


### 5.1 Vista rápida de las respuestas

Inspeccionamos una respuesta representativa de cada tipo de prompt para confirmar que el modelo está respondiendo en el formato esperado.


In [ ]:
for nombre in PROMPTS.keys():
    print("=" * 70)
    print(f"PROMPT: {nombre}")
    print("=" * 70)
    fila = df_resultados[df_resultados["prompt"] == nombre].iloc[0]
    print(fila["respuesta"])
    print()


---
## 6. Métricas objetivas

Calculamos cuatro métricas automáticas:

| Métrica | Cómo se calcula |
|---|---|
| **Etiqueta correcta (1/0)** | Buscamos `positiva` / `negativa` / `neutra` en la respuesta y comparamos con `ETIQUETA_GOLD = "negativa"`. |
| **Cumplimiento del formato (1/0)** | Comprobamos que aparezcan los marcadores que exige cada prompt (p. ej. `Clasificación:` y `Justificación:` en el prompt 2). |
| **Apartados solicitados (proporción)** | Cuántos de los apartados pedidos aparecen, sobre el total. |
| **Consistencia (% misma etiqueta)** | Sobre las 5 repeticiones del mismo prompt: porcentaje que coincide con la etiqueta más frecuente. |


In [ ]:
def _normaliza(s: str) -> str:
    s = s.lower()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return s


def extraer_etiqueta(respuesta: str):
    """
    Extrae la etiqueta (positiva / negativa / neutra) de la respuesta del modelo.
    Estrategia: primero buscamos un patrón explícito; si no, cogemos la primera etiqueta del texto.
    """
    texto = _normaliza(respuesta)

    # Patrones explícitos típicos
    patrones = [
        r"clasificacion(?:\s+final)?\s*[:\-]\s*\**\s*(positiva|negativa|neutra|neutral)",
        r"conclusion[^\n]*clasificacion[^\n]*[:\-]\s*\**\s*(positiva|negativa|neutra|neutral)",
        r"etiqueta\s*[:\-]\s*(positiva|negativa|neutra|neutral)",
    ]
    for pat in patrones:
        m = re.search(pat, texto)
        if m:
            v = m.group(1)
            return "neutra" if v.startswith("neutr") else v

    # Fallback: primera ocurrencia en texto libre
    m = re.search(r"\b(positiv[ao]|negativ[ao]|neutr[ao]|neutral)\b", texto)
    if m:
        v = m.group(1)
        if v.startswith("positiv"): return "positiva"
        if v.startswith("negativ"): return "negativa"
        return "neutra"
    return None


def cumple_formato(respuesta: str, prompt_id: str) -> int:
    t = _normaliza(respuesta)
    if prompt_id == "1_base":
        # El prompt base solo pide la clasificación; basta con que la etiqueta aparezca.
        return int(extraer_etiqueta(respuesta) is not None)
    if prompt_id == "2_plantilla":
        return int(("clasificacion" in t) and ("justificacion" in t))
    if prompt_id == "3_razonamiento":
        return int(
            ("aspectos positivos" in t)
            and ("aspectos negativos" in t)
            and ("alternativas" in t or "analisis de alternativas" in t)
            and ("conclusion" in t)
        )
    return 0


# Apartados esperados por prompt
APARTADOS = {
    "1_base": ["etiqueta_clasificacion"],
    "2_plantilla": ["clasificacion", "justificacion"],
    "3_razonamiento": [
        "aspectos positivos",
        "aspectos negativos",
        "analisis de alternativas",
        "conclusion",
    ],
}


def proporcion_apartados(respuesta: str, prompt_id: str) -> float:
    t = _normaliza(respuesta)
    requeridos = APARTADOS[prompt_id]
    if prompt_id == "1_base":
        return 1.0 if extraer_etiqueta(respuesta) else 0.0
    encontrados = sum(1 for r in requeridos if r in t)
    return round(encontrados / len(requeridos), 2)


# Aplicamos las métricas a cada fila
df_resultados["etiqueta_predicha"] = df_resultados["respuesta"].apply(extraer_etiqueta)
df_resultados["etiqueta_correcta"] = (df_resultados["etiqueta_predicha"] == ETIQUETA_GOLD).astype(int)
df_resultados["formato_ok"] = df_resultados.apply(lambda r: cumple_formato(r["respuesta"], r["prompt"]), axis=1)
df_resultados["apartados"] = df_resultados.apply(lambda r: proporcion_apartados(r["respuesta"], r["prompt"]), axis=1)

# Diagnóstico de fallos de extracción
no_extraidas = df_resultados["etiqueta_predicha"].isna().sum()
print(f"Etiquetas no extraídas: {no_extraidas}/{len(df_resultados)}")
if no_extraidas > 0:
    fallos = df_resultados[df_resultados["etiqueta_predicha"].isna()]
    print(f"\n⚠ Primeros ejemplos sin etiqueta:")
    for _, row in fallos.head(2).iterrows():
        print(f"\n  prompt={row['prompt']} | rep={row['repeticion']}")
        print(f"  Respuesta: {row['respuesta'][:300]}")

df_resultados[["prompt", "repeticion", "etiqueta_predicha", "etiqueta_correcta", "formato_ok", "apartados", "tiempo_s"]]


### 6.1 Consistencia entre repeticiones

Para cada prompt calculamos la proporción de repeticiones que coinciden con la etiqueta más frecuente. Cuanto más alto, más estable es el modelo bajo ese prompt.


In [ ]:
def consistencia(serie):
    """% de respuestas que coinciden con la moda."""
    if serie.empty:
        return 0.0
    moda = serie.mode().iloc[0] if not serie.mode().empty else None
    if moda is None:
        return 0.0
    return round((serie == moda).mean(), 2)


resumen = (
    df_resultados.groupby("prompt")
      .agg(
          n=("repeticion", "count"),
          acierto_etiqueta=("etiqueta_correcta", "mean"),
          formato_ok=("formato_ok", "mean"),
          apartados_medio=("apartados", "mean"),
          consistencia=("etiqueta_predicha", consistencia),
          etiqueta_moda=("etiqueta_predicha", lambda s: s.mode().iloc[0] if not s.mode().empty else None),
          tiempo_medio_s=("tiempo_s", "mean"),
      )
      .round(2)
)
resumen


---
## 7. Métricas subjetivas (escala 1-5)

Las métricas subjetivas las puntúa el grupo manualmente leyendo las respuestas. Para cada respuesta evaluamos:

- **Claridad** de la justificación.
- **Coherencia** del razonamiento.
- **Utilidad** de la respuesta para el caso de uso (mejorar atención al cliente).
- **Calidad argumentativa**.

A continuación se genera una **plantilla en blanco** que el grupo rellena con valores entre 1 y 5. Tras puntuar, se ejecuta la celda de agregación para obtener medias por prompt.

> Sustituid los `None` por enteros entre 1 y 5. Si queréis evaluar todas las repeticiones, podéis hacerlo; si no, basta con evaluar 2-3 representativas por prompt y dejar las demás a `None` (se ignoran al promediar).


In [ ]:
# Plantilla de evaluación subjetiva.
# Rellenad los valores con enteros 1-5. Los None se ignoran al promediar.

evaluacion_subjetiva = []
for _, fila in df_resultados.iterrows():
    evaluacion_subjetiva.append({
        "prompt": fila["prompt"],
        "repeticion": fila["repeticion"],
        "claridad": None,                # 1-5
        "coherencia": None,              # 1-5
        "utilidad": None,                # 1-5
        "calidad_argumentativa": None,   # 1-5
        "comentarios": "",
    })

df_subj = pd.DataFrame(evaluacion_subjetiva)
df_subj


### 7.1 Ejemplo orientativo (sustituir por puntuaciones reales)

Esta celda rellena automáticamente puntuaciones plausibles para que veáis cómo se calcula la agregación. **Para la entrega real, descartad esta celda y rellenad la anterior a mano.**


In [ ]:
# DEMO orientativa: rellena automáticamente puntuaciones plausibles.
# IMPORTANTE: descartad esta celda y rellenad la anterior a mano para la entrega real.

np.random.seed(42)

mapa_demo = {
    "1_base":          {"claridad": 3, "coherencia": 3, "utilidad": 2, "calidad_argumentativa": 2},
    "2_plantilla":     {"claridad": 4, "coherencia": 4, "utilidad": 4, "calidad_argumentativa": 4},
    "3_razonamiento":  {"claridad": 4, "coherencia": 5, "utilidad": 5, "calidad_argumentativa": 5},
}

for i, row in df_subj.iterrows():
    base = mapa_demo[row["prompt"]]
    for k, v in base.items():
        ruido = np.random.choice([-1, 0, 0, 0, 1])
        df_subj.at[i, k] = int(np.clip(v + ruido, 1, 5))

df_subj


In [ ]:
# Agregación de métricas subjetivas por prompt
campos_subj = ["claridad", "coherencia", "utilidad", "calidad_argumentativa"]
resumen_subj = (
    df_subj.groupby("prompt")[campos_subj]
           .mean()
           .round(2)
)
resumen_subj


---
## 8. Tabla comparativa final

Unimos todas las métricas en una única tabla por prompt para facilitar la comparación en la presentación.


In [ ]:
tabla_final = resumen.join(resumen_subj)
tabla_final


In [ ]:
# Visualización con barras
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Métricas objetivas
tabla_final[["acierto_etiqueta", "formato_ok", "apartados_medio", "consistencia"]].plot(
    kind="bar", ax=axes[0], rot=0
)
axes[0].set_title("Métricas objetivas por prompt — Phi-3-mini")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Proporción")
axes[0].legend(loc="lower right", fontsize=8)

# Métricas subjetivas
tabla_final[["claridad", "coherencia", "utilidad", "calidad_argumentativa"]].plot(
    kind="bar", ax=axes[1], rot=0
)
axes[1].set_title("Métricas subjetivas (1-5) por prompt — Phi-3-mini")
axes[1].set_ylim(0, 5.5)
axes[1].set_ylabel("Puntuación media")
axes[1].legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()


---
## 9. Análisis de variabilidad

Más allá de la consistencia de la **etiqueta**, observamos también la longitud de las respuestas y su dispersión. Con `temperature=0.7` esperamos cierta variación en cómo se redacta la justificación, pero un buen prompt debería estabilizar la **decisión final**.


In [ ]:
df_resultados["longitud_chars"] = df_resultados["respuesta"].str.len()
df_resultados["longitud_palabras"] = df_resultados["respuesta"].str.split().str.len()

variabilidad = (
    df_resultados.groupby("prompt")
      .agg(
          long_media_chars=("longitud_chars", "mean"),
          long_std_chars=("longitud_chars", "std"),
          long_media_palabras=("longitud_palabras", "mean"),
          long_std_palabras=("longitud_palabras", "std"),
          etiquetas_distintas=("etiqueta_predicha", lambda s: s.nunique()),
      )
      .round(1)
)
variabilidad


In [ ]:
# Tabla de etiquetas por repetición → de un vistazo se ve qué prompt es más estable
print("=== DISTRIBUCIÓN DE ETIQUETAS PREDICHAS POR PROMPT ===")
pd.crosstab(df_resultados["prompt"], df_resultados["etiqueta_predicha"], dropna=False)


---
## 10. Exportar resultados a CSV

Para llevar los resultados a la presentación, exportamos los DataFrames a CSV. Funciona tanto en Colab (descarga automática) como en local (los archivos quedan junto al notebook).


In [ ]:
import os

output_path_resultados = "resultados_phi3_mini.csv"
output_path_resumen    = "resumen_phi3_mini.csv"

df_resultados.to_csv(output_path_resultados, index=False, encoding="utf-8-sig")
tabla_final.to_csv(output_path_resumen, encoding="utf-8-sig")

print(f"✅ Resultados detallados : {os.path.abspath(output_path_resultados)}")
print(f"✅ Resumen por prompt    : {os.path.abspath(output_path_resumen)}")

# Si estamos en Colab, descarga automática; si no, los CSVs quedan en la carpeta del notebook.
try:
    from google.colab import files
    files.download(output_path_resultados)
    files.download(output_path_resumen)
except ImportError:
    print("\n(Ejecutándose en local: los CSVs se han guardado junto al notebook.)")


---
## 11. Conclusiones (rellenar tras la ejecución)

> Esta sección debe redactarse **después** de ejecutar el notebook, mirando los resultados reales obtenidos por Phi-3-mini en la sesión que entreguéis.

### 11.1 Resultados principales

**Exactitud de la clasificación:**
- Prompt base: ...
- Prompt plantilla: ...
- Prompt razonamiento: ...

**Consistencia entre repeticiones:**
- El modelo muestra mayor estabilidad con el prompt ...
- ...

### 11.2 Comparación entre estrategias de prompting

- El **prompt base** produce respuestas ... porque ...
- El **prompt plantilla** mejora/empeora ... porque ...
- El **prompt de razonamiento** destaca en ... aunque introduce ...

### 11.3 Comportamiento del modelo Phi-3-mini

- Fortalezas observadas: ...
- Debilidades observadas: ...
- Casos en los que el modelo falla sistemáticamente: ...

### 11.4 Comparación con los otros modelos del grupo

> Esta tabla se completa en la presentación común con los datos de Gemma 2 y Mistral 7B.

| Métrica | Gemma 2 (2,6B) | Mistral 7B (4-bit) | Phi-3-mini (3,8B) |
|---|---|---|---|
| Acierto etiqueta (medio) | ... | ... | ... |
| Cumplimiento formato | ... | ... | ... |
| Consistencia | ... | ... | ... |
| Tiempo medio por respuesta | ... | ... | ... |

### 11.5 Limitaciones del experimento

- Una sola reseña como caso de prueba: los resultados no son estadísticamente representativos.
- La evaluación subjetiva depende del criterio del evaluador.
- Los resultados varían entre ejecuciones por la naturaleza estocástica del muestreo (`temperature=0.7`).
- El modelo puede tener sesgos derivados de su entrenamiento que afectan a categorías específicas.

### 11.6 Reflexión final

> *(Completar con las conclusiones del grupo)*

---

### Referencia de parámetros

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `do_sample` | `True` | Muestreo estocástico (necesario para variabilidad) |
| `temperature` | `0.7` | Aleatoriedad moderada (requisito del enunciado: > 0) |
| `top_p` | `0.9` | Nucleus sampling |
| `max_new_tokens` | 200-400 | Más para el prompt 3 (razonamiento extendido) |
| `torch_dtype` | `float16` | Tipo recomendado para Phi-3 en T4 |
| `repeticiones` | 5 por prompt | Para estimar consistencia |
